In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_15923/1386256284.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()
len(docs)

9

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap = 100)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

28

In [5]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
)

In [6]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [7]:
query = "Machine Learnings and Data Science Content"
data = vector_store.similarity_search(query=query)

In [8]:
context = ""
for doc in data:
    context += doc.page_content + "\n"

In [9]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [10]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context":context,
        "question":query
    }

In [11]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answerd based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Question: {question}
""")

In [12]:
rag_chain = get_context | prompt | llm

In [13]:
res = rag_chain.invoke("What is the value of RBC Count and is it in range ? ")

In [14]:
print(res.content)

**RBC Count:** 4.47 million cells / mm³ (mill/mm³)  
**Reference range:** 3.80 – 4.80 mill/mm³  

**Result:** 4.47 is within the normal reference range.
